# Capstone: Predicting Titanic survival

**Question:** Can information known about a passenger before the voyage help us predict survival?

This notebook demonstrates a complete binary-classification workflow: define the question, inspect the data, split before learning preprocessing parameters, compare models with cross-validation, evaluate once on a held-out test set, and communicate limitations.

**Reproducibility:** all randomized steps use `RANDOM_STATE = 42`. Run the notebook from top to bottom.

## 1. Setup

We keep imports and display settings together. The warning filter only hides a harmless seaborn/pandas deprecation warning; model warnings remain visible.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

print("Random seed:", RANDOM_STATE)

## 2. Load and validate the data

The course uses seaborn's public Titanic CSV. Its column names are lowercase, so every field reference below is lowercase too. The assertions fail early if the remote schema changes.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(DATA_URL)

required_columns = {
    "survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"
}
assert required_columns.issubset(df.columns), "The Titanic schema has changed."
assert set(df["survived"].dropna().unique()) == {0, 1}

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print("Columns:", ", ".join(df.columns))
df.head()

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(1),
    "unique": df.nunique(dropna=True),
})
print("Exact duplicate rows:", df.duplicated().sum())
summary

### First observations

- `survived` is the target: 0 means died and 1 means survived.
- `age`, `fare`, and `embarked` contain missing values, so preprocessing must handle them.
- Columns such as `alive` directly restate the outcome and must never be predictors. `who` and `adult_male` also encode information already represented by age/sex and can obscure interpretation.

## 3. Exploratory data analysis

EDA is for understanding the sample, not for fitting transformations. These plots use the full dataset only descriptively. Model preprocessing is learned later from training data alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(data=df, x="survived", color="#4C78A8", ax=axes[0])
axes[0].set(title="Survival outcome", xlabel="Outcome", ylabel="Passengers")
axes[0].set_xticks([0, 1], labels=["Died", "Survived"])

sns.countplot(data=df, x="sex", hue="survived", palette="Set2", ax=axes[1])
axes[1].set(title="Survival counts by sex", xlabel="Sex", ylabel="Passengers")
handles, _ = axes[1].get_legend_handles_labels()
axes[1].legend(handles, ["Died", "Survived"], title="Outcome")

fig.suptitle("Titanic target and a key demographic pattern", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df.assign(outcome=df["survived"].map({0: "Died", 1: "Survived"}))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.histplot(
    data=plot_df, x="age", hue="outcome", bins=24,
    stat="density", common_norm=False, element="step", ax=axes[0]
)
axes[0].set(title="Age distributions by outcome", xlabel="Age (years)", ylabel="Density")

sns.boxplot(data=plot_df, x="pclass", y="fare", hue="outcome", palette="Set2", ax=axes[1])
axes[1].set(title="Fare by passenger class and outcome", xlabel="Passenger class", ylabel="Fare")
axes[1].set_ylim(0, df["fare"].quantile(0.95))
axes[1].legend(title="Outcome")

fig.suptitle("Numeric feature patterns (fare axis trimmed at the 95th percentile)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
survival_rates = (
    df.groupby(["pclass", "sex"], observed=True)["survived"]
    .agg(passengers="size", survival_rate="mean")
    .reset_index()
)
survival_rates["survival_rate"] = survival_rates["survival_rate"].round(3)
survival_rates

The plots show associations, not causes. Passenger class and sex were related to survival in this historical sample, but the model cannot tell us why. It also should not be used to make decisions about people in another setting.

## 4. Define predictors, then split

We use only variables plausibly available for a passenger and intentionally exclude outcome proxies (`alive`) and post-outcome/redundant engineered fields. We split **before** imputation, encoding, and scaling. Stratification keeps the target proportion similar in both partitions.

In [ ]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]
feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["survived"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "survival_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])

assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)
split_summary.round(3)

## 5. Build leakage-safe pipelines

`ColumnTransformer` learns medians, most-frequent categories, one-hot levels, and scaling statistics only when `.fit()` is called. Because it sits inside each model pipeline, cross-validation learns those values separately inside every training fold.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

models = {
    "Logistic regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]),
    "Decision tree": Pipeline([
        ("preprocess", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE
        )),
    ]),
}

print("Predictors:", feature_columns)
print("Excluded leakage example: alive")

## 6. Compare models using training data only

Five-fold stratified cross-validation estimates performance without touching the held-out test set. Accuracy is intuitive here; ROC AUC also checks ranking quality. The standard deviation shows how much the estimate varies across folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
        n_jobs=1,
    )
    rows.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_sd": scores["test_accuracy"].std(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("cv_roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)
cv_results.round(3)

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_accuracy = baseline.score(np.zeros((len(y_test), 1)), y_test)

best_model_name = cv_results.loc[0, "model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

print(f"Most-frequent baseline accuracy: {baseline_accuracy:.3f}")
print("Selected from cross-validation:", best_model_name)

## 7. Final evaluation on untouched test data

Now—and only now—we use the test set. This is one final estimate, not another opportunity to tune the model.

In [ ]:
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Test accuracy: {test_accuracy:.3f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Died", "Survived"],
    digits=3,
    zero_division=0,
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["Died", "Survived"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Held-out confusion matrix")

RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1], color="#E45756")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Chance")
axes[1].set_title("Held-out ROC curve")
axes[1].legend(loc="lower right")

fig.suptitle(f"{best_model_name}: final test evaluation", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 8. Interpret the fitted model carefully

Permutation importance measures how much test accuracy falls when one raw feature is shuffled. It is model-specific and reflects predictive usefulness, not causality. Correlated predictors can share importance.

In [ ]:
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="accuracy",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance_mean": importance_result.importances_mean,
        "importance_sd": importance_result.importances_std,
    })
    .sort_values("importance_mean", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(
    importance["feature"],
    importance["importance_mean"],
    xerr=importance["importance_sd"],
    color="#72B7B2",
    alpha=0.9,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Permutation importance on held-out data", xlabel="Mean decrease in accuracy", ylabel="Feature")
plt.tight_layout()
plt.show()

importance.sort_values("importance_mean", ascending=False).round(3)

## 9. Conclusions and limitations

The final outputs above support a concise project summary:

1. Compare the selected model's held-out accuracy with the majority-class baseline.
2. Use the class-specific recall in the report, not accuracy alone.
3. Treat permutation importance as predictive association, never as proof of cause.
4. Remember that this is one historical passenger list with missing values and social inequities; results do not automatically generalize.

**Possible next steps:** tune hyperparameters using nested cross-validation, evaluate calibration, engineer family-size/title features without using the target, and repeat evaluation across several fixed splits.

In [ ]:
top_feature = importance.sort_values("importance_mean", ascending=False).iloc[0]["feature"]
print(f"Selected model: {best_model_name}")
print(f"Held-out accuracy: {test_accuracy:.3f} (baseline: {baseline_accuracy:.3f})")
print(f"Highest permutation importance in this split: {top_feature}")
print("Titanic capstone complete: data -> EDA -> split -> CV -> final test -> interpretation.")